# 🌌 APEX Launcher

Welcome to the **APEX** (Adaptive Platform for Unified AI Platform Configuration, Orchestration and Workspace Management) launcher. This notebook contains the **Stage-0 Bootstrap** loader which clones the GitHub repository, configures system search paths, installs dependencies, and boots the runtime dashboard.

### Stage-0 Bootstrap Sequence
1. **Environment Verification**: Detects if running on Google Colab or local systems.
2. **Google Drive Mount**: Prompts for Google Drive authentication in Colab if persistent storage is desired.
3. **Repository Clone**: Clones the codebase into target storage if missing.
4. **Path & Package Injection**: Updates `sys.path` to enable importing the runtime modules.
5. **Dependency Installation**: Runs pip installer requirements checks before loading application code.

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

# 1. Clear sys.modules import cache to prevent stale memory imports in Colab
for mod in list(sys.modules.keys()):
    if mod.startswith("bootstrap") or mod.startswith("runtime"):
        del sys.modules[mod]

is_colab = "google.colab" in sys.modules
repo_url = "https://github.com/Nikhil-Kumar-Shah/APEX.git"
default_branch = "main"

# Default local target path
target_dir = Path.cwd().parent / "APEX"

if is_colab:
    print("[+] Google Colab environment detected.")
    # Mount Google Drive for persistent setup if desired
    mount_drive = input("Mount Google Drive for persistent storage? (y/n) [y]: ").strip().lower() != 'n'
    if mount_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            print("[+] Google Drive mounted successfully for persistent workspaces.")
        except Exception as e:
            print(f"[-] Google Drive mount failed: {e}. Falling back to temporary storage.")
    
    # Always use Colab VM for repository and runtime execution
    target_dir = Path("/content/APEX")

print(f"[+] Target Installation Directory: {target_dir}")

# 2. Existing Repository Resolution Mode
if (target_dir / ".git").exists():
    print("\n[!] Existing repository detected. Choose sync mode:")
    print("  [1] Synchronize/Update working tree (Default)")
    print("  [2] Fresh Re-clone (Deletes existing folder and clones fresh)")
    print("  [3] Offline/Use As-Is (No updates)")
    choice = input("Enter choice [1]: ").strip()
    if choice == "2":
        print(f"[+] Deleting existing repository at {target_dir}...")
        shutil.rmtree(target_dir, ignore_errors=True)
    elif choice == "3":
        print("[+] Offline mode selected. Using existing workspace as-is.")
    else:
        print("[+] Syncing repository working tree with origin/main...")
        try:
            subprocess.run(["git", "-C", str(target_dir), "fetch", "origin"], check=True)
            subprocess.run(["git", "-C", str(target_dir), "checkout", default_branch], check=True)
            subprocess.run(["git", "-C", str(target_dir), "reset", "--hard", f"origin/{default_branch}"], check=True)
            subprocess.run(["git", "-C", str(target_dir), "clean", "-fd"], check=True)
            print("[+] Sync completed successfully.")
        except subprocess.CalledProcessError as e:
            print(f"[-] Network sync failed: {e}. Using existing workspace as-is.")

# 3. Fresh Clone if deleted or not present
if not target_dir.exists() or not (target_dir / ".git").exists():
    print(f"[+] Repository not found. Cloning from {repo_url}...")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", repo_url, str(target_dir)], check=True)

# 4. Update Python System Search Paths
repo_path_str = str(target_dir.resolve())
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)
else:
    sys.path.remove(repo_path_str)
    sys.path.insert(0, repo_path_str)

# 5. Diagnostic Version Report
print("\n" + "=" * 50)
print("             APEX Environment Diagnostics")
print("=" * 50)
try:
    branch = subprocess.run(["git", "-C", str(target_dir), "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
    commit = subprocess.run(["git", "-C", str(target_dir), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    tags = subprocess.run(["git", "-C", str(target_dir), "tag"], capture_output=True, text=True).stdout.strip().split()
    
    print(f"Repository Path: {target_dir.resolve()}")
    print(f"Current Branch : {branch}")
    print(f"Current Commit : {commit}")
    print(f"Git Tags       : {tags if tags else 'None'}")
except Exception as e:
    print(f"[-] Git Diagnostics failed: {e}")

try:
    from bootstrap.version_manager import VersionManager
    from bootstrap.config import REQUIRED_MANIFEST
    import runtime
    v_mgr = VersionManager()
    print(f"Bootstrap Version: {v_mgr.get_checkout_ref(target_dir)}")
    print(f"Runtime Version  : {getattr(runtime, '__version__', '1.0.0')}")
    print(f"Manifest Targets : {REQUIRED_MANIFEST}")
except Exception as e:
    print(f"[-] Module Diagnostics failed: {e}")
print("=" * 50 + "\n")

# 6. Legacy code checks (Self-Diagnostics)
print("[+] Running repository self-diagnostics for legacy patterns...")
legacy_detected = False
for p in target_dir.rglob("*"):
    if p.is_file() and p.suffix in [".py", ".md", ".json"] and ".git" not in p.parts and "__pycache__" not in p.parts:
        try:
            content = p.read_text(encoding="utf-8")
            if "v1.0.0" in content and "version_manager.py" in p.name:
                print(f"  [!] Found legacy tag ref 'v1.0.0' in: {p.relative_to(target_dir)}")
                legacy_detected = True
            if "configs" in content and "repository_manager.py" in p.name:
                print(f"  [!] Found legacy hardcoded folder 'configs' in: {p.relative_to(target_dir)}")
                legacy_detected = True
        except Exception:
            pass
if not legacy_detected:
    print("[+] Self-diagnostics completed. No active legacy patterns found.")

# 7. Install requirements dependencies
requirements_file = target_dir / "requirements.txt"
if requirements_file.is_file():
    print("\n[+] Installing Python dependencies from requirements.txt...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)], check=True)

# 8. Import and hand over to Stage-1 Bootstrap
try:
    from bootstrap.installer import InstallationWizard
    from bootstrap.launcher import RuntimeLauncher

    print("\n[+] Initializing Stage-1 Installation Wizard...")
    # Set interactive=False inside the notebook to auto-confirm if non-interactive run
    wizard = InstallationWizard(workspace_parent_dir=target_dir.parent, repo_url=repo_url)
    project_path = wizard.run(interactive=False)

    if project_path:
        launcher = RuntimeLauncher(project_path)
        launcher.launch()
    else:
        print("[-] Stage-1 Installation Wizard failed.")
except ImportError as e:
    print(f"[-] Failed to import bootstrap module: {e}. Check folder integrity.")